# PCGT Colab Runner (Drive Data, No Internet Download)

This notebook is cleaned for a simple workflow:

1. Clone or update the repo.
2. Install Python dependencies only (no dataset downloads).
3. Mount Google Drive once.
4. Use dataset directly from Drive via symlink.
5. Verify GPU.
6. Run either a quick test or all 7 datasets.

Assumption: your dataset is already present at `/content/drive/MyDrive/PCGT/data` with the same structure as local `data/`.

## 1) Clone Or Update Repository

Run this once per session to ensure code is current.

In [16]:
import os

if not os.path.exists('/content/PCGT'):
    !git clone https://github.com/ranjanchoubey/PCGT.git /content/PCGT
else:
    %cd /content/PCGT
    !git pull
    %cd /content

%cd /content/PCGT

## 2) Install Dependencies

This installs code dependencies only. It does not download datasets.

In [27]:
import torch

# Uninstall previous versions to prevent conflicts
!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster --y

# Install compatible versions using the PyG index URL
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install get-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Optionally, install PyTorch Geometric itself
!pip install git+https://github.com/pyg-team/pytorch_geometric.git


In [ ]:
! pip install torch-sparse

## 3) Mount Drive And Use Data In-Place

This mounts Drive and creates `/content/PCGT/data` as a symlink to your Drive dataset.

No data copy is performed.

In [17]:
import os
from datetime import datetime
from google.colab import drive

# 1) Mount drive once
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=False)
else:
    print('Drive already mounted.')

# 2) Your uploaded folder in Drive
DRIVE_UPLOAD_ROOT = '/content/drive/MyDrive/PCGT_datasets'

# Supports either:
# - /content/drive/MyDrive/PCGT_datasets/data/...
# - /content/drive/MyDrive/PCGT_datasets/... (data files directly inside)
if os.path.isdir(os.path.join(DRIVE_UPLOAD_ROOT, 'data')):
    DRIVE_DATA_ROOT = os.path.join(DRIVE_UPLOAD_ROOT, 'data')
else:
    DRIVE_DATA_ROOT = DRIVE_UPLOAD_ROOT

if not os.path.isdir(DRIVE_DATA_ROOT):
    print('Could not find dataset root at:', DRIVE_DATA_ROOT)
    print('Top-level folders in MyDrive:')
    !ls -la /content/drive/MyDrive | head -n 50
    raise RuntimeError('Fix DRIVE_UPLOAD_ROOT and rerun this cell.')

LOCAL_DATA_ROOT = '/content/PCGT/data'
FORCE_DRIVE_LINK = True

# 3) Point /content/PCGT/data to Drive data (no copy)
if os.path.islink(LOCAL_DATA_ROOT):
    current = os.path.realpath(LOCAL_DATA_ROOT)
    if current != DRIVE_DATA_ROOT:
        os.remove(LOCAL_DATA_ROOT)
        os.symlink(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
elif not os.path.exists(LOCAL_DATA_ROOT):
    os.symlink(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
elif os.path.isdir(LOCAL_DATA_ROOT):
    if FORCE_DRIVE_LINK:
        backup_dir = f"{LOCAL_DATA_ROOT}_local_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        os.rename(LOCAL_DATA_ROOT, backup_dir)
        os.symlink(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
        print('Moved existing local directory to:', backup_dir)
    else:
        print(f'Using existing directory at {LOCAL_DATA_ROOT} (not a symlink).')
        print('Set FORCE_DRIVE_LINK=True to replace it with a Drive symlink.')
else:
    raise RuntimeError(f'Path exists and is not a directory/symlink: {LOCAL_DATA_ROOT}')

print('Drive upload root:', DRIVE_UPLOAD_ROOT)
print('Drive data root  :', DRIVE_DATA_ROOT)
print('Local data path  :', LOCAL_DATA_ROOT, '->', os.path.realpath(LOCAL_DATA_ROOT))

!ls -la /content/PCGT/data | head -n 20

## 5) Verify GPU

Run this before training.

In [18]:
import torch
print('torch:', torch.__version__)
print('cuda :', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi | head -n 15

## 6) Optional Full Run (All 7 Datasets)

Runs all final configs and saves logs to Drive.

In [24]:
%cd /content/PCGT/medium

import os
from datetime import datetime

DATA_DIR = '/content/PCGT/data/'  # keep trailing slash
RESULTS_DIR = '/content/drive/MyDrive/PCGT/results_colab'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f"{RESULTS_DIR}/{RUN_TAG}"
os.makedirs(RUN_DIR, exist_ok=True)

print('Data dir   :', DATA_DIR)
print('Results dir:', RUN_DIR)

# Cora

In [ ]:
!python -B main.py --method pcgt --dataset cora --lr 0.01 --num_layers 4 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 7 --graph_weight 0.8 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 2>&1 | tee {RUN_DIR}/cora_pcgt_gpu.txt

# CiteSeer


In [ ]:
!python -B main.py --method pcgt --dataset citeseer --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 7 --graph_weight 0.7 --dropout 0.5 --weight_decay 0.01 --ours_weight_decay 0.02 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/citeseer_pcgt_gpu.txt

# PubMed


In [ ]:
!python -B main.py --method pcgt --dataset pubmed --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 50 --graph_weight 0.8 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/pubmed_pcgt_gpu.txt

# Chameleon

In [ ]:
!python -B main.py --method pcgt --dataset chameleon --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 10 --graph_weight 0.8 --dropout 0.5 --weight_decay 0.001 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/chameleon_pcgt_gpu.txt

# Film

In [ ]:
!python -B main.py --method pcgt --dataset film --lr 0.05 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --seed 123 --device 0 --epochs 500 --patience 200 --runs 10 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 5 --graph_weight 0.5 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/film_pcgt_gpu.txt

# Squirrel

In [33]:
!python -B main.py --method pcgt --dataset squirrel --lr 0.01 --num_layers 4 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 10 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 10 --graph_weight 0.8 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/squirrel_pcgt_gpu.txt

# Deezer

In [29]:
%cd /content/PCGT/medium
import os

DZ_DIR = f"{RUN_DIR}/deezer_attack"
os.makedirs(DZ_DIR, exist_ok=True)

# Common args for all Deezer runs
BASE = (
    "python -B main.py --method pcgt --dataset deezer-europe "
    "--use_graph --use_residual --backbone gcn --rand_split --seed 123 "
    f"--device 0 --epochs 500 --patience 200 --runs 3 --display_step 100 --aggregate add --data_dir {DATA_DIR}"
)

# ──────────────────────────────────────────────────────────────
# DZ-B1: Capacity reduction — hidden=32, reps=2
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B1: hidden=32, reps=2 ===")
!{BASE} --hidden_channels 32 --num_layers 2 --ours_layers 1 --num_reps 2 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-5 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B1_h32_reps2.txt

# ──────────────────────────────────────────────────────────────
# DZ-B2: Even smaller — hidden=32, reps=1, ours_layers=1
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B2: hidden=32, reps=1, minimal ===")
!{BASE} --hidden_channels 32 --num_layers 2 --ours_layers 1 --num_reps 1 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.6 --weight_decay 1e-4 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B2_h32_reps1.txt

# ──────────────────────────────────────────────────────────────
# DZ-B3: GCN layers=1 (simpler backbone)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B3: num_layers=1, h=64 ===")
!{BASE} --hidden_channels 64 --num_layers 1 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-5 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B3_L1.txt

In [30]:
# ──────────────────────────────────────────────────────────────
# DZ-B4: Heavy dropout=0.7, stronger wd
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B4: dropout=0.7, wd=5e-3 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B4_d07_wd5e3.txt

# ──────────────────────────────────────────────────────────────
# DZ-B5: Very heavy dropout=0.8 + wd=0.01
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B5: dropout=0.8, wd=0.01 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.8 --weight_decay 0.01 --ours_weight_decay 0.05 --ours_dropout 0.6 \
  2>&1 | tee {DZ_DIR}/DZ-B5_d08_wd01.txt

# ──────────────────────────────────────────────────────────────
# DZ-B6: Moderate dropout=0.7 + small hidden=32
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B6: h=32, d=0.7, wd=5e-3 ===")
!{BASE} --hidden_channels 32 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B6_h32_d07.txt

In [32]:
# ──────────────────────────────────────────────────────────────
# DZ-B7: BatchNorm + gw=0.9 (heavy GCN)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B7: use_bn, gw=0.9 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.9 --use_bn \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B7_bn_gw09.txt

# ──────────────────────────────────────────────────────────────
# DZ-B8: BatchNorm + gw=0.3 (heavy PCGT)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B8: use_bn, gw=0.3 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.3 --use_bn \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B8_bn_gw03.txt

# ──────────────────────────────────────────────────────────────
# DZ-B9: KMeans K=20 (larger partitions, more nodes per cluster)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B9: kmeans K=20 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 20 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B9_K20.txt

# ──────────────────────────────────────────────────────────────
# DZ-B10: Random partitioning (ablation — are KMeans partitions actually helping?)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B10: random partition K=50 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method random --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B10_random.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# DZ-B11: No feature normalization
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B11: no_feat_norm ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 --no_feat_norm \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B11_nofeatnorm.txt

# ──────────────────────────────────────────────────────────────
# DZ-B12: Pure GCN baseline (no PCGT attention at all)
# graph_weight=1.0 means output = 1.0*GCN + 0.0*PCGT
# This tells us the GCN ceiling on Deezer
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B12: pure GCN (gw=1.0) ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 1.0 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B12_pure_gcn.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# SUMMARY: Parse all Deezer attack results
# ──────────────────────────────────────────────────────────────
import re, glob

print("="*70)
print("  DEEZER ATTACK RESULTS SUMMARY")
print("="*70)
print(f"{'Config':<30} {'Highest Test':<20} {'Final Test':<20}")
print("-"*70)

results = []
for f in sorted(glob.glob(f"{DZ_DIR}/DZ-B*.txt")):
    name = os.path.basename(f).replace('.txt','')
    with open(f) as fh:
        text = fh.read()
    m = re.search(r'Highest Test:\s+([\d.]+)\s*±\s*([\d.]+)', text)
    m2 = re.search(r'Final Test:\s+([\d.]+)\s*±\s*([\d.]+)', text)
    if m:
        ht = f"{m.group(1)} ± {m.group(2)}"
        ft = f"{m2.group(1)} ± {m2.group(2)}" if m2 else "N/A"
        results.append((name, float(m.group(1)), ht, ft))
        print(f"{name:<30} {ht:<20} {ft:<20}")
    else:
        print(f"{name:<30} {'PARSING ERROR':<20}")

if results:
    best = max(results, key=lambda x: x[1])
    print("-"*70)
    print(f"BEST: {best[0]} → {best[2]}")
    print(f"\nPrevious best: 64.94 ± 0.85  |  SGFormer target: 67.1 ± 1.1")

## After the sweep: Run the best config with 5 runs

Copy the best config from above and change `--runs 3` to `--runs 5` below.

In [ ]:
# ──────────────────────────────────────────────────────────────
# BEST CONFIG — 5 runs (fill in the winning config from above)
# Example: if DZ-B4 won, paste its args here with --runs 5
# ──────────────────────────────────────────────────────────────

# !{BASE} --hidden_channels ?? --num_layers 2 --ours_layers 1 --num_reps ?? \
#   --partition_method kmeans --num_partitions ?? --graph_weight ?? \
#   --lr 0.01 --dropout ?? --weight_decay ?? --ours_weight_decay ?? --ours_dropout ?? \
#   --runs 5 \
#   2>&1 | tee {DZ_DIR}/DZ-BEST_5run.txt